In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
api_key = os.environ["ANTHROPIC_API_KEY"]
print(f"Key loaded")

Key loaded


In [2]:
from anthropic import Anthropic

client = Anthropic()

response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say 'API works' and nothing else."}],
)
print(response.content[0].text)
print(f"Tokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

API works
Tokens: 17 in, 5 out


In [2]:
import time
from pathlib import Path

import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from sampling import _train_thought_pool, sample_random_examples, sample_category_examples
from prompts import build_prompt

# Config
load_dotenv()
SPLIT_DIR = Path("../../processed")
OUT_DIR = Path("../../models/06_reframing/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL = "claude-sonnet-4-6"
EXPERIMENTS = ["E1", "E2", "E3", "E4"]

client = Anthropic()

### Load splits and prepare test set
Collapse the 113 test rows to 57 unique thoughts and builds the multi-reference lookup that scoring will use later.

In [3]:
train_df = pd.read_csv(SPLIT_DIR / "cognitive_reframing_train.csv")
test_df_raw = pd.read_csv(SPLIT_DIR / "cognitive_reframing_test.csv")

# Deduplicate test to one row per unique thought_id (first occurrence)
test_df = (
    test_df_raw
    .drop_duplicates(subset="thought_id", keep="first")
    .reset_index(drop=True)
)

# Build the multi-reference lookup: thought_id -> list of reframes
references_by_thought = (
    test_df_raw
    .groupby("thought_id")["reframe"]
    .apply(list)
    .to_dict()
)

print(f"Train pool: {len(train_df)} rows, {train_df['thought_id'].nunique()} unique thoughts")
print(f"Test set:   {len(test_df)} unique thoughts (from {len(test_df_raw)} rows)")
print(f"\nReference reframes per test thought:")
ref_counts = pd.Series({tid: len(refs) for tid, refs in references_by_thought.items()})
print(ref_counts.value_counts().sort_index().to_dict(),
      "  (key = #refs, value = #thoughts with that many refs)")

Train pool: 467 rows, 226 unique thoughts
Test set:   57 unique thoughts (from 113 rows)

Reference reframes per test thought:
{1: 1, 2: 56}   (key = #refs, value = #thoughts with that many refs)


In [4]:
train_pool = _train_thought_pool(train_df)
print(f"Train pool collapsed: {len(train_pool)} unique thoughts available for example sampling")
print(f"Per-category counts in pool:")
print(train_pool["category"].value_counts())

Train pool collapsed: 226 unique thoughts available for example sampling
Per-category counts in pool:
category
all-or-nothing thinking        31
labeling                       29
negative feeling or emotion    29
mind reading                   26
catastrophizing                20
fortune telling                17
emotional reasoning            14
overgeneralizing               14
disqualifying the positive     14
personalizing                  13
blaming                         8
should statements               6
comparing and despairing        5
Name: count, dtype: int64


In [5]:
from api_caller import call_api

output_path = OUT_DIR / "experiment_outputs.csv"

# Headers
all_rows = []

print(f"Running {len(EXPERIMENTS)} experiments × {len(test_df)} test thoughts = "
      f"{len(EXPERIMENTS) * len(test_df)} API calls")
print(f"Estimated time: ~{len(EXPERIMENTS) * len(test_df) * 3 / 60:.0f} minutes at ~3s/call\n")

total_input_tokens = 0
total_output_tokens = 0
t_start = time.perf_counter()

for i, row in test_df.iterrows():
    tid = row["thought_id"]
    cat = row["category"]

    # Sample examples once per test thought (reused across experiments)
    e2_examples = sample_random_examples(train_pool, tid, n=3, salt="random")
    e3e4_examples, fallback = sample_category_examples(train_pool, tid, cat, n=3, salt="category")

    # Run all 4 experiments for this thought
    for exp_id in EXPERIMENTS:
        if exp_id == "E1":
            prompt = build_prompt(exp_id, thought=row["thought"])
            ex_ids = []
        elif exp_id == "E2":
            prompt = build_prompt(exp_id, thought=row["thought"], examples=e2_examples)
            ex_ids = [ex.thought for ex in e2_examples]  # store thought text as identifier proxy
        elif exp_id == "E3":
            prompt = build_prompt(exp_id, thought=row["thought"], category=cat, examples=e3e4_examples)
            ex_ids = [ex.thought for ex in e3e4_examples]
        elif exp_id == "E4":
            prompt = build_prompt(
                exp_id, thought=row["thought"], situation=row["situation"],
                category=cat, examples=e3e4_examples,
            )
            ex_ids = [ex.thought for ex in e3e4_examples]

        result = call_api(client, prompt)

        all_rows.append({
            "experiment_id": exp_id,
            "thought_id": tid,
            "category": cat,
            "situation": row["situation"],
            "thought": row["thought"],
            "reference_reframe": row["reframe"],  # primary ref (one of possibly many)
            "generated_reframe": result["text"],
            "example_thought_1": ex_ids[0] if len(ex_ids) >= 1 else "",
            "example_thought_2": ex_ids[1] if len(ex_ids) >= 2 else "",
            "example_thought_3": ex_ids[2] if len(ex_ids) >= 3 else "",
            "category_fallback": fallback if exp_id in ("E3", "E4") else "",
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "latency_ms": result["latency_ms"],
            "error": result["error"],
        })

        total_input_tokens += result["input_tokens"]
        total_output_tokens += result["output_tokens"]

    # Incremental save every 10 thoughts
    if (i + 1) % 10 == 0:
        pd.DataFrame(all_rows).to_csv(output_path, index=False)
        elapsed = time.perf_counter() - t_start
        print(f"  [{i + 1}/{len(test_df)}] thoughts done | "
              f"elapsed {elapsed:.0f}s | "
              f"~{elapsed / (i + 1) * (len(test_df) - i - 1):.0f}s remaining")

# Final save
pd.DataFrame(all_rows).to_csv(output_path, index=False)

# Cost estimate
input_cost = total_input_tokens / 1_000_000 * 3.0
output_cost = total_output_tokens / 1_000_000 * 15.0
total_elapsed = time.perf_counter() - t_start

print(f"\n=== Done ===")
print(f"Total time: {total_elapsed / 60:.1f} min")
print(f"Total tokens: {total_input_tokens} in, {total_output_tokens} out")
print(f"Estimated cost: ${input_cost + output_cost:.3f} "
      f"(${input_cost:.3f} input + ${output_cost:.3f} output)")
print(f"Output saved to: {output_path}")

Running 4 experiments × 57 test thoughts = 228 API calls
Estimated time: ~11 minutes at ~3s/call

  [10/57] thoughts done | elapsed 90s | ~425s remaining
  [20/57] thoughts done | elapsed 185s | ~342s remaining
  [30/57] thoughts done | elapsed 273s | ~245s remaining
  [40/57] thoughts done | elapsed 365s | ~155s remaining
  [50/57] thoughts done | elapsed 461s | ~64s remaining

=== Done ===
Total time: 8.8 min
Total tokens: 83380 in, 9278 out
Estimated cost: $0.389 ($0.250 input + $0.139 output)
Output saved to: ../../models/06_reframing/results/experiment_outputs.csv


In [6]:
df_out = pd.read_csv(output_path)

print(f"Total rows in output: {len(df_out)} (expected: {len(EXPERIMENTS) * len(test_df)})")
print(f"\nBy experiment:")
print(df_out["experiment_id"].value_counts().sort_index())

print(f"\nErrors:")
errors = df_out[df_out["error"].notna()]
print(f"  {len(errors)} error rows")
if len(errors) > 0:
    print(errors[["experiment_id", "thought_id", "error"]].head())

print(f"\nEmpty outputs:")
empty = df_out[df_out["generated_reframe"] == ""]
print(f"  {len(empty)} empty generations")

print(f"\nCategory-matching fallbacks (E3/E4):")
fallback_rows = df_out[df_out["category_fallback"].str.startswith("partial", na=False)]
print(f"  {len(fallback_rows)} rows used partial fallback")
if len(fallback_rows) > 0:
    print(fallback_rows[["experiment_id", "category", "category_fallback"]].value_counts())

print(f"\nSample of E4 outputs:")
print(df_out[df_out["experiment_id"] == "E4"][["thought", "generated_reframe"]].head(3).to_string())

Total rows in output: 228 (expected: 228)

By experiment:
experiment_id
E1    57
E2    57
E3    57
E4    57
Name: count, dtype: int64

Errors:
  0 error rows

Empty outputs:
  0 empty generations

Category-matching fallbacks (E3/E4):
  0 rows used partial fallback

Sample of E4 outputs:
                                                         thought                                                                                                                                                                   generated_reframe
3                                        he took me for granted!                                                            He may have simply forgotten to return it, and that doesn't necessarily mean he sees me as someone to take advantage of.
7           We won't be able to find the store before it closes.  Getting lost doesn't mean we won't find the store in time — we still have options like asking for directions or using a map, and it's possible we'll figure 